# 08 - Expand GAEZ Suitability to 10 Crops

The original 4 crops (rice, maize, cassava, soybean) were a reasonable Assignment I scope, but for
real analysis they're too thin: maize and soybean never come close to winning anywhere in Banten
(see notebook 07's validation), so the effective result was really a 2-crop story. This adds 6 more
crops - chosen to span Banten's actual landscape niches (coastal, lowland, upland), not picked
arbitrarily:

| Code | Crop | Niche |
|---|---|---|
| OLP | Oil palm | Lowland, high rainfall |
| RUB | Para-rubber | Upland, well-drained |
| SUC | Sugar cane | Lowland/coastal |
| COF | Coffee | Upland (Pandeglang/Lebak hills) |
| COC | Cacao | Upland |
| CON | Coconut | Coastal |

Codes verified against the official GAEZ v5 crop list (Appendix 4-1, Module V sheet - the sheet
matching Theme 4/RES05, the suitability variable this project uses), not guessed. Same `HILM`
(irrigated, high input) scenario as the existing 4 crops, for a fair, consistent comparison.

In [1]:
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\preprocessing


## 1. Download the 6 new crops under HILM (irrigated, high input)

In [2]:
from common import RAW_DIR, PROCESSED_DIR, ensure_dir, get_logger, download_file

log = get_logger("expand_gaez_crops")

BASE_URL_TMPL = "https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET/{variable}/"
GAEZ_DIR = RAW_DIR / "gaez_v5"
ensure_dir(GAEZ_DIR)

PERIOD = "HP0120"
CLIMATE_SOURCE = "AGERA5"
SSP = "HIST"
WATER_INPUT_IRRIGATED = "HILM"

NEW_CROP_CODES = {
    "oilpalm": "OLP", "rubber": "RUB", "sugarcane": "SUC",
    "coffee": "COF", "cacao": "COC", "coconut": "CON",
}
VARIABLES = {"RES05-SIX": "suitability_class", "RES05-SXX": "suitability_index"}

irrigated_paths = {}
for crop, code in NEW_CROP_CODES.items():
    for variable, label in VARIABLES.items():
        dest = GAEZ_DIR / f"{label}_{crop}_{code}_irrigated.tif"
        fname = f"GAEZ-V5.{variable}.{PERIOD}.{CLIMATE_SOURCE}.{SSP}.{code}.{WATER_INPUT_IRRIGATED}.tif"
        url = BASE_URL_TMPL.format(variable=variable) + fname
        download_file(url, dest)
        irrigated_paths[(crop, label)] = dest

print("Done -", len(irrigated_paths), "new rasters ready in", GAEZ_DIR)

[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_oilpalm_OLP_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_oilpalm_OLP_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_rubber_RUB_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_rubber_RUB_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_sugarcane_SUC_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_sugarcane_SUC_irrigated.tif


... (3 outputs left out)

[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_cacao_COC_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_coconut_CON_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_coconut_CON_irrigated.tif


Done - 12 new rasters ready in D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5


## 2. Zonal statistics onto the existing 1 km grid

In [3]:
import logging

import geopandas as gpd
from rasterstats import zonal_stats

logging.getLogger("rasterio._env").setLevel(logging.ERROR)

GRID_PATH = PROCESSED_DIR / "grid" / "grid_1km.gpkg"
grid = gpd.read_file(GRID_PATH)
grid_4326 = grid.to_crs("EPSG:4326")
print(f"{len(grid)} grid cells loaded")

new_values = {}
for crop in NEW_CROP_CODES:
    class_stats = zonal_stats(
        grid_4326.geometry, str(irrigated_paths[(crop, "suitability_class")]),
        stats=["majority"], all_touched=True,
    )
    index_stats = zonal_stats(
        grid_4326.geometry, str(irrigated_paths[(crop, "suitability_index")]),
        stats=["mean"], all_touched=True,
    )
    new_values[(crop, "class")] = [s["majority"] for s in class_stats]
    new_values[(crop, "index")] = [s["mean"] for s in index_stats]
    n_valid = sum(v is not None for v in new_values[(crop, "index")])
    print(f"{crop}: {n_valid}/{len(grid)} cells got a value")

D:\Softwares\Anaconda\Miniconda\envs\geoai-suitability\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
D:\Softwares\Anaconda\Miniconda\envs\geoai-suitability\Lib\site-packages\rasterio\env.py:664: RuntimeWarning: Cannot find gdalvrt.xsd (GDAL_DATA is not defined)
  elif GDALDataFinder().find_file("gdalvrt.xsd"):


9333 grid cells loaded


oilpalm: 9333/9333 cells got a value


rubber: 9333/9333 cells got a value


sugarcane: 9333/9333 cells got a value


coffee: 9333/9333 cells got a value


cacao: 9333/9333 cells got a value


coconut: 9333/9333 cells got a value


## 3. Add new columns to `tabular_features.csv` (idempotent)

In [4]:
import pandas as pd

FEATURES_PATH = PROCESSED_DIR / "features" / "tabular_features.csv"
df = pd.read_csv(FEATURES_PATH)
assert len(df) == len(grid), "row count mismatch between features table and grid"

added = []
for crop in NEW_CROP_CODES:
    col = f"gaez_index_{crop}"
    if col in df.columns:
        print(f"{crop}: column already present - skipping")
        continue
    df[f"gaez_class_{crop}"] = new_values[(crop, "class")]
    df[f"gaez_index_{crop}"] = new_values[(crop, "index")]
    added.append(crop)

df.to_csv(FEATURES_PATH, index=False)
print(f"Wrote {FEATURES_PATH} - {len(df)} rows, {len(df.columns)} columns")
print("Crops added this run:", added)

Wrote D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\processed\features\tabular_features.csv - 9333 rows, 57 columns
Crops added this run: ['oilpalm', 'rubber', 'sugarcane', 'coffee', 'cacao', 'coconut']


## 4. Validate: full 10-crop label distribution

In [5]:
import numpy as np

ALL_CROPS = ["rice", "maize", "cassava", "soybean", "oilpalm", "rubber", "sugarcane", "coffee", "cacao", "coconut"]
NOT_RECOMMENDED_THRESHOLD = 5000
TIE_THRESHOLD = 500


def label_cells(frame, crops):
    idx_cols = [f"gaez_index_{c}" for c in crops]
    scores = frame[idx_cols].to_numpy()
    order = np.argsort(scores, axis=1)
    best_i, second_i = order[:, -1], order[:, -2]
    best_score = scores[np.arange(len(frame)), best_i]
    second_score = scores[np.arange(len(frame)), second_i]
    names = np.array([c.capitalize() for c in crops])
    labels = names[best_i].astype(object)
    tie_mask = (best_score - second_score) <= TIE_THRESHOLD
    labels[tie_mask] = [
        " / ".join(sorted((names[b], names[s]))) + " (tied)"
        for b, s in zip(best_i[tie_mask], second_i[tie_mask])
    ]
    labels[best_score < NOT_RECOMMENDED_THRESHOLD] = "Not recommended"
    return pd.Series(labels, index=frame.index)


new_label = label_cells(df, ALL_CROPS)
print("Distribution with all 10 crops:")
print((new_label.value_counts(normalize=True) * 100).round(1))

print()
print("Max index reached by each crop, anywhere in Banten:")
for c in ALL_CROPS:
    print(f"  {c}: max={df[f'gaez_index_{c}'].max():.0f}, mean={df[f'gaez_index_{c}'].mean():.0f}")

lat_tertiles = pd.qcut(df["lat"], 3, labels=["South (upland)", "Central", "North (coastal)"])
print()
print("Mean suitability index per crop, by zone:")
print(df.groupby(lat_tertiles, observed=True)[[f"gaez_index_{c}" for c in ALL_CROPS]].mean().round(0).T)

Distribution with all 10 crops:
Not recommended             54.7
Coconut                     15.7
Oilpalm                     11.3
Coconut / Oilpalm (tied)    10.0
Coffee / Oilpalm (tied)      4.7
Cassava / Oilpalm (tied)     3.5
Coconut / Coffee (tied)      0.2
Name: proportion, dtype: float64

Max index reached by each crop, anywhere in Banten:
  rice: max=8800, mean=3039
  maize: max=4021, mean=1315
  cassava: max=9026, mean=3960
  soybean: max=5548, mean=1735
  oilpalm: max=9395, mean=4395
  rubber: max=3320, mean=2255
  sugarcane: max=5505, mean=2344
  coffee: max=9395, mean=3557
  cacao: max=7277, mean=3013
  coconut: max=9970, mean=3926

Mean suitability index per crop, by zone:
lat                   South (upland)  Central  North (coastal)
gaez_index_rice               1784.0   2683.0           4650.0
gaez_index_maize               836.0   1129.0           1978.0
gaez_index_cassava            2645.0   3921.0           5313.0
gaez_index_soybean            1005.0   1658.0        

## Notes

- 6 new crops added, all under `HILM` (irrigated, high input), same scenario as the original 4.
- No `_rainfed` audit columns for these 6 - they were never downloaded under `HRLM` in the first
  place (unlike the original 4, which started there before notebook 07's fix).
- Dashboard (`scripts/dashboard/app.py`) needs its `CROPS` list updated to pick these up - separate step.